# GardnAIze — 05 ResNet-18 transfer learning

Three-class garden-photo quality gate. Uses a pretrained torchvision ResNet-18, freezes the backbone, trains the final classification layer, records loss/accuracy, and exports weights **outside Git**. No performance claims until a real licensed dataset is collected and evaluated.

Install locally: `pip install torch torchvision pillow pandas scikit-learn matplotlib`.

The model's pretrained weights may be downloaded on first use. Verify license and model-card terms before redistribution.

In [ ]:
import json, random, copy
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
DATA=ROOT/"datasets"/"images"/"garden_validation"
OUT=ROOT/"models"/"weights"
OUT.mkdir(parents=True,exist_ok=True)
assert DATA.exists(), f"Missing labeled dataset: {DATA}"
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

In [ ]:
weights=models.ResNet18_Weights.DEFAULT
train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean=weights.transforms().mean,std=weights.transforms().std)])
eval_tf=weights.transforms()
raw=datasets.ImageFolder(DATA)
assert set(raw.classes)=={"garden","not_garden","unusable"}, f"Unexpected classes: {raw.classes}"
labels=[y for _,y in raw.samples]
assert min(np.bincount(labels))>=5, "Collect at least 5 examples per class (more strongly recommended)."
idx=np.arange(len(labels))
train_idx,temp_idx=train_test_split(idx,test_size=0.30,stratify=labels,random_state=SEED)
temp_labels=np.array(labels)[temp_idx]
val_idx,test_idx=train_test_split(temp_idx,test_size=0.5,stratify=temp_labels,random_state=SEED)
print("Train / validation / test:",len(train_idx),len(val_idx),len(test_idx))
print("Important: replace random splitting with group-based splitting for images of the same garden/photographer.")
train_ds=datasets.ImageFolder(DATA,transform=train_tf)
eval_ds=datasets.ImageFolder(DATA,transform=eval_tf)
train_loader=DataLoader(Subset(train_ds,train_idx.tolist()),batch_size=8,shuffle=True,num_workers=0)
val_loader=DataLoader(Subset(eval_ds,val_idx.tolist()),batch_size=8,shuffle=False,num_workers=0)
test_loader=DataLoader(Subset(eval_ds,test_idx.tolist()),batch_size=8,shuffle=False,num_workers=0)

In [ ]:
model=models.resnet18(weights=weights)
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,len(raw.classes))
model=model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.fc.parameters(),lr=1e-3)
EPOCHS=5
history=[]; best_acc=-1.0; best_state=None
for epoch in range(EPOCHS):
    metrics={}
    for phase,loader in [("train",train_loader),("val",val_loader)]:
        model.train(phase=="train")
        running_loss=correct=total=0
        for images,targets in loader:
            images,targets=images.to(device),targets.to(device)
            with torch.set_grad_enabled(phase=="train"):
                logits=model(images)
                loss=criterion(logits,targets)
                if phase=="train":
                    optimizer.zero_grad();loss.backward();optimizer.step()
            running_loss+=loss.item()*targets.size(0)
            correct+=(logits.argmax(1)==targets).sum().item();total+=targets.size(0)
        metrics[phase+"_loss"]=running_loss/max(total,1)
        metrics[phase+"_accuracy"]=correct/max(total,1)
    history.append({"epoch":epoch+1,**metrics})
    print(history[-1])
    if metrics["val_accuracy"]>best_acc:
        best_acc=metrics["val_accuracy"];best_state=copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
torch.save(model.state_dict(),OUT/"garden_resnet18_v1.pth")
(OUT/"garden_resnet18_v1.json").write_text(json.dumps({"architecture":"resnet18","classes":raw.classes,"pretrained_weights":str(weights),"seed":SEED,"validation_accuracy":best_acc,"note":"Preliminary: random split; group-based evaluation required before deployment."},indent=2))
print("Saved locally:",OUT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
metrics_df=pd.DataFrame(history)
display(metrics_df)
metrics_df.plot(x="epoch",y=["train_loss","val_loss"],title="Cross-entropy loss")
plt.show()

Do not treat a random split as a production-grade evaluation: group by garden/photographer before reporting results to the jury. Keep test data unseen until final evaluation. For a CPU-only demo, the model still runs.